# 叠加小模型

Happy MI：动手学机制可解释性

训练五个稀疏输入特征经过二维瓶颈重构。先预测降低共同激活概率会改变什么。

正文与图示 CC BY-NC-SA 4.0；代码 MIT。实现由 experiments/superposition.py 自动生成。

## 环境

以下单元会联网安装固定教学依赖。如果运行时提示重启，请重启后从下一节开始。推荐 Python 3.11。

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "numpy==2.3.3", "torch==2.8.0"])

In [ ]:
import random
import numpy as np
import torch

def seed_all(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.set_num_threads(1)


## 看懂实现

先读函数，标出输入、被观察或替换的变量、评价指标。此单元只定义函数。

In [ ]:
"""Train the tied-weight ReLU toy model on independent sparse features."""

import json
import torch


class ToyModel(torch.nn.Module):
    def __init__(self, features=5, hidden=2):
        super().__init__()
        self.W = torch.nn.Parameter(torch.randn(features, hidden) * 0.2)
        self.bias = torch.nn.Parameter(torch.zeros(features))

    def forward(self, x):
        return torch.relu((x @ self.W) @ self.W.T + self.bias)


def sparse_batch(batch, features, probability, generator=None):
    values = torch.rand(batch, features, generator=generator)
    active = torch.rand(batch, features, generator=generator) < probability
    return values * active


def train_one(probability, seed=42, steps=1500):
    seed_all(seed)
    model = ToyModel()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    for _ in range(steps):
        x = sparse_batch(256, 5, probability)
        loss = ((model(x) - x) ** 2).sum(dim=-1).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    # Independent stream: test data must not depend on training step count.
    test_generator = torch.Generator().manual_seed(seed + 10000)
    with torch.no_grad():
        test = sparse_batch(4096, 5, probability, test_generator)
        error = (model(test) - test) ** 2
        mse, baseline = error.mean().item(), test.square().mean().item()
        unit = torch.nn.functional.normalize(model.W, dim=-1)
        return {
            'probability': probability, 'seed': seed, 'steps': steps,
            'test_mse_per_coordinate': mse, 'zero_baseline_mse': baseline,
            'relative_mse': mse / baseline,
            'active_coordinate_mse': error[test > 0].mean().item(),
            'weights': model.W.tolist(), 'bias': model.bias.tolist(),
            'norms': model.W.norm(dim=-1).tolist(),
            'cosine_matrix': (unit @ unit.T).tolist(),
        }


def run(seed=42):
    return {'kind': 'trained_synthetic', 'features': 5, 'hidden': 2,
            'runs': [train_one(p, seed) for p in [0.5, 0.1, 0.03]]}



## 运行并检查

保留原始结果，不只保留符合预期的部分。

In [ ]:
result = run()
print(json.dumps(result, ensure_ascii=False, indent=2))

## 练习

运行 run(seed=7)，比较方向、零基线与活跃特征误差。不要只看绝对 MSE。

写下：哪些结果支持你的预测？哪些不支持？这个实验不能说明什么？